In [1]:
!pip uninstall -y protobuf tensorflow tensorboard
!pip install --no-cache-dir --upgrade protobuf tensorflow baseballcv ultralytics statcast-pitches "numpy<2.0.0" -q

print("✅ NUCLEAR FIX COMPLETE. 🛑 NOW GO RESTART YOUR RUNTIME 🛑")

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tensorboard 2.20.0
Uninstalling tensorboard-2.20.0:
  Successfully uninstalled tensorboard-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 193.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 201.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 316.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 274.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 246.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 288.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ⬇️ CELL 1: LOAD THE MEGA ARSENAL ⬇️
from baseballcv.functions import LoadTools
from ultralytics import YOLO
import torch

print("Loading the full Broadcast Arsenal...")
load_tools = LoadTools()

phc_model = YOLO(load_tools.load_model("phc_detector"))
ball_model = YOLO(load_tools.load_model("ball_trackingv4"))
bat_model = YOLO(load_tools.load_model("bat_tracking"))
glove_model = YOLO(load_tools.load_model("glove_tracking"))
pose_model = YOLO('yolo26l-pose.pt')

print("✅ All 5 Models Loaded & Ready for the Ultimate Pipeline!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading the full Broadcast Arsenal...


Model downloaded to models/YOLO/pitcher_hitter_catcher_detector/model_weights/pitcher_hitter_catcher_detector_v4.pt


Model downloaded to models/YOLO/ball_tracking/model_weights/ball_trackingv4.pt


Model downloaded to models/YOLO/bat_tracking/model_weights/bat_tracking.pt


Model downloaded to models/YOLO/glove_tracking/model_weights/glove_tracking.pt
✅ All 5 Models Loaded & Ready for the Ultimate Pipeline!


In [3]:
import cv2
import numpy as np

input_video = 'test-video.mp4'
output_video = 'perfect_visual_broadcast_with_contact.mp4'

cap = cv2.VideoCapture(input_video)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps == 0: fps = 30

out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

SKELETON_CONNECTIONS = [(5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]
sz_top_smooth, sz_bottom_smooth, hp_left_smooth, hp_right_smooth = 0, 0, 0, 0
contact_display_frames = 0

frame_count = 0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Igniting Visual Pipeline with Bat Tracking. {total_frames} frames to process...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_count += 1

    if frame_count % 30 == 0:
        print(f"Progress: {frame_count}/{total_frames} frames")

    # ==========================================
    # 1. PLAYERS (Blue Boxes + Labels)
    # ==========================================
    phc_results = phc_model(frame, verbose=False, conf=0.50)
    target_boxes = []

    if phc_results[0].boxes is not None:
        for box in phc_results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            conf = box.conf[0].cpu().item()
            cls_name = phc_model.names[int(box.cls[0].cpu().item())].lower()

            # Draw Blue Box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

            # Draw Label (Solid blue background, white text)
            label = f"{cls_name} {conf:.2f}"
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1 - 20), (x1 + w, y1), (255, 0, 0), -1)
            cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            if cls_name in ["pitcher", "hitter"]:
                target_boxes.append((x1, y1, x2, y2))

    # ==========================================
    # 2. SKELETONS (Green Lines + Red Dots)
    # ==========================================
    pose_results = pose_model(frame, verbose=False, conf=0.50)
    current_sz_top, current_sz_bottom = None, None

    if pose_results[0].keypoints is not None and len(target_boxes) > 0:
        largest_height = 0
        for keypoints in pose_results[0].keypoints.xy.cpu().numpy():
            valid_pts = [pt for pt in keypoints if pt[0] > 0]
            if not valid_pts: continue

            center_x = sum(pt[0] for pt in valid_pts) / len(valid_pts)
            center_y = sum(pt[1] for pt in valid_pts) / len(valid_pts)

            is_target = any(px1 < center_x < px2 and py1 < center_y < py2 for (px1, py1, px2, py2) in target_boxes)

            if is_target:
                for kp1_idx, kp2_idx in SKELETON_CONNECTIONS:
                    pt1, pt2 = keypoints[kp1_idx], keypoints[kp2_idx]
                    if pt1[0] > 0 and pt2[0] > 0:
                        cv2.line(frame, (int(pt1[0]), int(pt1[1])), (int(pt2[0]), int(pt2[1])), (0, 255, 0), 2)

                for pt in keypoints:
                    if pt[0] > 0 and pt[1] > 0:
                        cv2.circle(frame, (int(pt[0]), int(pt[1])), 4, (0, 0, 255), -1)

                h = max(pt[1] for pt in valid_pts) - min(pt[1] for pt in valid_pts)
                if h > largest_height:
                    largest_height = h
                    s_y = [keypoints[i][1] for i in [5,6] if keypoints[i][1] > 0]
                    h_y = [keypoints[i][1] for i in [11,12] if keypoints[i][1] > 0]
                    k_y = [keypoints[i][1] for i in [13,14] if keypoints[i][1] > 0]
                    if s_y and h_y and k_y:
                        current_sz_top = (sum(s_y)/len(s_y) + sum(h_y)/len(h_y)) / 2
                        current_sz_bottom = sum(k_y)/len(k_y)

    # ==========================================
    # 3. BAT, BALL, AND CONTACT DETECTION
    # ==========================================
    ball_box_coords, bat_box_coords = None, None

    # Track Ball (Yellow Box + Label)
    ball_results = ball_model(frame, verbose=False, conf=0.15)
    if ball_results[0].boxes is not None and len(ball_results[0].boxes) > 0:
        box = ball_results[0].boxes[0]
        bx1, by1, bx2, by2 = map(int, box.xyxy[0].cpu().numpy())
        conf = box.conf[0].cpu().item()
        ball_box_coords = (bx1, by1, bx2, by2)

        cv2.rectangle(frame, (bx1, by1), (bx2, by2), (0, 255, 255), 2)
        label = f"ball {conf:.2f}"
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (bx1, by1 - 20), (bx1 + w, by1), (0, 255, 255), -1)
        cv2.putText(frame, label, (bx1, by1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    # Track Bat (Purple Box + Label)
    bat_results = bat_model(frame, verbose=False, conf=0.20)
    if bat_results[0].boxes is not None and len(bat_results[0].boxes) > 0:
        box = bat_results[0].boxes[0]
        batx1, baty1, batx2, baty2 = map(int, box.xyxy[0].cpu().numpy())
        conf = box.conf[0].cpu().item()
        bat_box_coords = (batx1, baty1, batx2, baty2)

        cv2.rectangle(frame, (batx1, baty1), (batx2, baty2), (255, 0, 255), 2)
        label = f"bat {conf:.2f}"
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (batx1, baty1 - 20), (batx1 + w, baty1), (255, 0, 255), -1)
        cv2.putText(frame, label, (batx1, baty1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # Collision Math
    if ball_box_coords and bat_box_coords:
        bx1, by1, bx2, by2 = ball_box_coords
        batx1, baty1, batx2, baty2 = bat_box_coords
        buffer = 15
        if bx1 < (batx2 + buffer) and bx2 > (batx1 - buffer) and by1 < (baty2 + buffer) and by2 > (baty1 - buffer):
            contact_display_frames = 15

    # Flash CONTACT text
    if contact_display_frames > 0:
        cv2.putText(frame, "CONTACT!", (width//2 - 150, int(height*0.3)), cv2.FONT_HERSHEY_DUPLEX, 3.0, (0, 0, 255), 5)
        contact_display_frames -= 1

    # ==========================================
    # 4. STRIKE ZONE (White Box Only)
    # ==========================================
    glove_results = glove_model(frame, verbose=False, conf=0.30)
    hp_box = None

    if glove_results[0].boxes is not None:
        for box in glove_results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            cls_name = glove_model.names[int(box.cls[0].cpu().item())].lower()
            if "home" in cls_name or "plate" in cls_name:
                hp_box = (x1, y1, x2, y2)

    if current_sz_top and current_sz_bottom and hp_box:
        if sz_top_smooth == 0:
            sz_top_smooth, sz_bottom_smooth = current_sz_top, current_sz_bottom
            hp_left_smooth, hp_right_smooth = hp_box[0], hp_box[2]
        else:
            sz_top_smooth = 0.8 * sz_top_smooth + 0.2 * current_sz_top
            sz_bottom_smooth = 0.8 * sz_bottom_smooth + 0.2 * current_sz_bottom
            hp_left_smooth = 0.8 * hp_left_smooth + 0.2 * hp_box[0]
            hp_right_smooth = 0.8 * hp_right_smooth + 0.2 * hp_box[2]

        pt1 = (int(hp_left_smooth), int(sz_top_smooth))
        pt2 = (int(hp_right_smooth), int(sz_bottom_smooth))

        cv2.rectangle(frame, pt1, pt2, (255, 255, 255), 2)

    out.write(frame)

cap.release()
out.release()
print("FINAL VISUAL PIPELINE COMPLETE! ⚾")

Igniting Visual Pipeline with Bat Tracking. 393 frames to process...
Progress: 30/393 frames
Progress: 60/393 frames
Progress: 90/393 frames
Progress: 120/393 frames
Progress: 150/393 frames
Progress: 180/393 frames
Progress: 210/393 frames
Progress: 240/393 frames
Progress: 270/393 frames
Progress: 300/393 frames
Progress: 330/393 frames
Progress: 360/393 frames
Progress: 390/393 frames
FINAL VISUAL PIPELINE COMPLETE! ⚾
